# SaliencySpark
This is a python library for converting saliency maps to Pytorch Tensors.

In [1]:
# install dependencies
# specify the device to use
USING_GPU_IF_AVAILABLE = True
import tqdm
import torch
import os
_ = torch.empty(1)
if torch.cuda.is_available() and USING_GPU_IF_AVAILABLE:
    _ = _.cuda()
DEVICE = _.device
print(f'[DEVICE={DEVICE}]')

/home/zhaojizhang/miniconda3/envs/openmmlab/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[DEVICE=cuda:0]


In [2]:
from PIL import Image
import torchvision.transforms.functional as TF
import pathlib
IMAGENET_RGB_MEAN = torch.tensor((0.27), device=DEVICE).reshape(1, 1, 1, 1) # torch.tensor((0.485, 0.456, 0.406), device=DEVICE).reshape(1, 3, 1, 1)
IMAGENET_RGB_STD  = torch.tensor((0.27), device=DEVICE).reshape(1, 1, 1, 1) # torch.tensor((0.229, 0.224, 0.225), device=DEVICE).reshape(1, 3, 1, 1)
def load_image(size: int, img_file: str):
    img = Image.open(img_file).convert('L')
    img = TF.resize(img, [size,size])
    # img = TF.center_crop(TF.resize(img, size), [size, size])
    img = TF.to_tensor(img).unsqueeze(0).to(DEVICE).sub(IMAGENET_RGB_MEAN).div_(IMAGENET_RGB_STD)
    return img
def denormalize(img_bchw):
    return img_bchw.mul(IMAGENET_RGB_STD).add_(IMAGENET_RGB_MEAN).clamp_(0., 1.)

video_folder = '/mnt/e/Li Lab/MS-TCA/MyMSTCN/clipped_sm'
tgt_npz_folder = '/mnt/e/Li Lab/MS-TCA/MyMSTCN/npz_sms'
tgt_path = pathlib.Path(tgt_npz_folder)
tgt_path.mkdir(parents=True, exist_ok=True)
video_folder_list = os.listdir(video_folder)
video_folder_list = [video_folder_name for video_folder_name in video_folder_list if os.path.isdir(os.path.join(video_folder, video_folder_name))]
video_folder_list[0]
# video_folder_list
# ['v_-B_LfcvJ_ow_111_128',
#  'v_-B_LfcvJ_ow_148_162',
#  'v_mCWca9iJcj0_122_138',
#  ...]
print(len(video_folder_list))

1898


In [4]:
from models.resnet18 import SaliencyMap
from timm.models import create_model

dense_encoder = create_model('SaliencyMap_INF')# SaliencyMap()
dense_encoder.load_state_dict(torch.load(os.path.join('..','ResNet18_nonMGF_800epoch','SaliencyMap_1kpretrained_timm_style.pth')))
# dense_encoder.to_dense()
# convert_sparse_bn_to_regular_bn(dense_encoder)
dense_encoder.eval()
dense_encoder = dense_encoder.cuda()
print(dense_encoder)


SaliencyMapConvNet_INF(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): GELU()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act2): GELU()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): GELU()
 

In [5]:
def process_folder_image(folder_path, folder_root, tgt_path):
    """
    open a folder, process all image in the folder, and save them into a npz file
    """
    tgt_image_names = os.listdir(os.path.join(folder_root, folder_path))
    tgt_image_names = [os.path.join(folder_root, folder_path, name) for name in tgt_image_names if name.endswith('.jpg') or name.endswith('.png')]
    rsts = [dense_encoder(load_image(224, name),False).flatten().unsqueeze(dim=0).cpu().detach() for name in tgt_image_names]
    #print(images[0])
    #print(images[0].shape)
    #rst = dense_encoder(images[0], True)
    #print([rst_.shape for rst_ in rst])
    rsts = torch.concat(rsts).numpy()
    # print(rsts.shape)
    rsts.tofile(os.path.join(tgt_path, folder_path + '.npz'))
    torch.cuda.empty_cache()

# process_folder_image(video_folder_list[0], video_folder, '')

In [6]:
for video_folder_lst in tqdm.tqdm(video_folder_list, total = len(video_folder_list)):
    process_folder_image(video_folder_lst, video_folder, tgt_npz_folder)

100%|███████████████████████████████████████████████████████████████████████████████| 1898/1898 [39:47<00:00,  1.26s/it]
